<a href="https://colab.research.google.com/github/jvch0/03MIAR---Algoritmos-de-Optimizacion---2026/blob/Trabajo-Practico/Copia_de_Trabajo_Pr%C3%A1ctico_Algoritmos(V2%2Cno_borrar).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Abner Jesua Chica Herrera  <br>
Url: https://github.com/jvch0/03MIAR---Algoritmos-de-Optimizacion---2026/blob/Trabajo-Practico/Copia_de_Trabajo_Pr%C3%A1ctico_Algoritmos(V2%2Cno_borrar).ipynb<br>
Google Colab: https://colab.research.google.com/drive/1OClx7nS75HhOVfvnqsm63-f2c0ZZUDtN?usp=sharing <br>
Problema:
>1. Sesiones de doblaje <br>

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:

Número de actores:  10

Número de tomas  :   30

Actores/Tomas :
https://bit.ly/36D8IuK

- 1 indica que el actor participa en la toma
- 0 en caso contrario







                                        

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

En el problema, el espacio de soluciones son todas las posibles combinaciones en las que se pueden repartir las tomas en determinados dias (se pautó 5 días).

La función objetivo para el problema es minimizar el número total de convocatoria de actores a lo largo de los días, matemáticamente representada como:

$$ Min Z = \sum_{d \epsilon D}\sum_{j \epsilon J}y_{jd}$$

donde:

- Z es la funcion a minimizar
- D son las sesiones disponibles, donde $$d  \epsilon \{1,...,D_{max}\}$$
- J son el conjunto de actores, donde $$j \epsilon \{1,...10\}  $$
- $y_{jd} \epsilon \{1,0\}$ donde Es 1 si el actor $j$ es convocado en el día $d$, y 0 en caso contrario


Las restricciones se plantearon de la siguiente manera:

- Cada toma debe programarse un día

$$ \sum_{d\epsilon D} x_{id} = 1, \forall i \epsilon I $$

- No pueden grabarse mas de 6 tomas por dia

$$\sum_{i \epsilon I} x_{id} \leq 6,  \forall d \epsilon D$$

- Si la toma de $i$ se graba el dia $d$, y el actor $j$actua en la toma, el actor $j$ debe convocarse

$$ y_{jd} \geq A_{ij} \cdot x_{ij}, \forall i \epsilon I, \forall j \epsilon J, \forall d \epsilon D$$

donde:
- $A_{ij}$ es la matriz binaria, donde $A_{ij}=1$ si la toma $i$ requiere al actor $j$
- $I$ es el conjunto de tomas donde $i \epsilon \{1,...,30\}$

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

La ecuacion para calcular las particiones de n elementos en k grupos de tamño s es:

$$P =  \frac{n!}{(s!)^k \cdot k!}$$

Sabiendo que $n=30 , k=5, s=6$ el resultado es del orden de $10^{16}$, es decir existen una cantidad de combinaciones posibles de un orden de $10^{16}$

Por fuerza bruta este algoritmo tendria una compejidad de $O(N!)$, pero con el algorimo de recocido simulado se tiene un orden de $O(I \cdot N \cdot A)$ es decir el tiempo aumenta de manera lineal con el numero de iteraciones, la cantidad de tomas y la cantidad de actores.
<br>
<br>
<br>

Se utilizo la LLM gemini para encontrar la ecuacion que permita realizar la agrupacion de elementos en k grupos de un tamaño s. el prompt usado fue "Existe alguna ecuación que me arroje la cantidad de posibles combinaciones al agrupar una cierta cantidad de elementos en una cantidad determinadad de grupos y de un tamaño en especifico"

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

Se utilizo el recocido simulado, el cual es un metodo basado en busqueda estocastica, la razón por la que se utilizó esta tecnica es porque ayuda a evitar minimos locales usando esa propiedad de en ciertas ocaciones tomar respuestas "malas" las cuales ayudan a salir de minimos locales, además de esto el espacio de soluciones es realmente extenso así que resolverlo por algún metodo de fuerza bruta tomaría mucho tiempo y este algoritmo no necesita revisar todas las opciones, ya que recurre a una especie de "caminata inteligente" a traves del espacio de soluciones

In [25]:
import pandas as pd
import numpy as np
import random
import math

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
def cargar_datos(ruta_archivo):
    df = pd.read_csv(ruta_archivo, skiprows=1)
    # Seleccionamos las 30 tomas y las columnas de los 10 actores
    matriz_actores = df.iloc[0:30, 1:11].values
    return matriz_actores

In [28]:
def calcular_costo(estado, matriz_tomas):
    costo_total = 0
    for dia in estado:
        if not dia: continue
        tomas_dia = matriz_tomas[dia]
        # np.any identifica si el actor actúa al menos una vez ese día
        actores_convocados = np.any(tomas_dia, axis=0).sum()
        costo_total += actores_convocados
    return costo_total

In [29]:
def generar_vecino(estado, max_tomas):

    #Esto crea deepcopy
    nuevo_estado = [dia[:] for dia in estado]
    num_dias = len(nuevo_estado)

    #Eleccion para cambiar el horario 50/50
    if random.random() < 0.5:
        # mueve una toma de un día a otro con espacio
        dias_tomas = [d for d in range(num_dias) if len(nuevo_estado[d]) > 0]
        dia_origen = random.choice(dias_tomas)
        dias_espacio = [d for d in range(num_dias) if len(nuevo_estado[d]) < max_tomas and d != dia_origen]

        #cambia las tomas que hay en los dias
        #por ejemplo agarra un dia de 6 tomas y otro de 4 a dos de 5 tomas
        if dias_espacio:
            dia_destino = random.choice(dias_espacio)
            toma = random.choice(nuevo_estado[dia_origen])
            nuevo_estado[dia_origen].remove(toma)
            nuevo_estado[dia_destino].append(toma)
    else:
        # Intercambiar dos tomas de días distintos hace un swap
        dia1, dia2 = random.sample(range(num_dias), 2)
        if nuevo_estado[dia1] and nuevo_estado[dia2]:
            idx1, idx2 = random.randrange(len(nuevo_estado[dia1])), random.randrange(len(nuevo_estado[dia2]))
            nuevo_estado[dia1][idx1], nuevo_estado[dia2][idx2] = nuevo_estado[dia2][idx2], nuevo_estado[dia1][idx1]

    return nuevo_estado

In [30]:
def s_a(matriz_tomas, num_dias=5, max_tomas=6, temp_inicial=100.0, tasa_enfriamiento=0.99, iteraciones_per_temp=100):
    num_tomas = matriz_tomas.shape[0]
    tomas_mezcladas = list(range(num_tomas))
    #Desordena las tomas
    random.shuffle(tomas_mezcladas)

    #Reparte tomas al azar en los 5 días
    estado_actual = [tomas_mezcladas[i*max_tomas : (i+1)*max_tomas] for i in range(num_dias)]
    costo_actual = calcular_costo(estado_actual, matriz_tomas)

    m_estado = [dia[:] for dia in estado_actual]
    m_costo = costo_actual

    # Listas para guardar el historial y graficar
    h_costos = []
    h_mejores = []

    temp = temp_inicial
    while temp > 0.01:
        for _ in range(iteraciones_per_temp):
            estado_vecino = generar_vecino(estado_actual, max_tomas)
            costo_vecino = calcular_costo(estado_vecino, matriz_tomas)

            delta = costo_vecino - costo_actual

            # Decisión de aceptación
            if delta < 0 or random.random() < math.exp(-delta / temp): #Si el cambio es bueno se acepta, si es malo se acepta aveces
                estado_actual, costo_actual = estado_vecino, costo_vecino
                if costo_actual < m_costo:
                    m_estado = [dia[:] for dia in estado_actual] #Aqui se guarda el mejor estado encontrado
                    m_costo = costo_actual

            h_costos.append(costo_actual)
            h_mejores.append(m_costo)
        temp *= tasa_enfriamiento #tasa de enfriamiento

    return m_estado, m_costo, h_costos, h_mejores

In [31]:
matriz = cargar_datos('/content/drive/MyDrive/Datos problema doblaje(30 tomas, 10 actores) - Hoja 1.csv')
mejor_h, mejor_c, h_costos, h_mejores =s_a(matriz)


print(f"Resultado final: {mejor_c} convocatorias.")
print(f"Costo Mínimo (Total de convocatorias): {mejor_c}")
for i, dia in enumerate(mejor_h):
    print(f"Día {i+1}: Tomas {dia}")

Resultado final: 27 convocatorias.
Costo Mínimo (Total de convocatorias): 27
Día 1: Tomas [20, 17, 23, 16, 13, 22]
Día 2: Tomas [27, 1, 29, 18, 26, 19]
Día 3: Tomas [28, 11, 9, 21, 25, 7]
Día 4: Tomas [6, 5, 24, 15, 8, 12]
Día 5: Tomas [0, 4, 3, 14, 2, 10]
